# 01 Hypothesis validation

## Adrian Vazquez 

--- 

# 01_hypothesis_validation

### <b> Adrián Vazquez </b>

---

1. <b> Hipótesis 1: Leader realmente lidera: </b>

Para cada par: 

$return_{leader}(t)  →  return_{follower}(t+1, t+2, t+3...) $ 

Pregunta: Cuando el leader se mueve, ¿el follower se mueve después?

2. <b> Hipótesis 2: La correlación 60 días sirve </b>

Compara: Top correlation pairs vs random same-sector pairs

Pregunta: ¿Los pares top 50 tienen mejor relación lead-lag que pares aleatorios?

3. <b> Hipótesis 3: Market cap como leader tiene sentido.  </b>

Compara dos versiones: 

  -  Leader = mayor market cap
  - Leader = ticker que estadísticamente lidera más

Pregunta: ¿Market cap predice liderazgo o sólo es una regla intuitiva?

4. <b> Hipótesis 4: Los edge factors tienen sentido. </b>

aun no optimizar. Solo medir frecuencia

Pregunta:
 - ¿Cuántas señales genera?
 - ¿Son demasiado raras?
 - ¿O demasiado frecuentes?

5. <b> Hipótesis 5: La señal tiene drift posterior.</b>

Después de señal: follower forward return 5m, 15m, 30m, 60m

Pregunta: 
 - Después de la señal, ¿el follower sube más que en condiciones normales? <- Esta es la prueba más importante antes del backtest

 <b> Output que hay que buscar </b>

 | Hipótesis        |                        Métrica | Resultado          | Decisión              |
| ---------------- | -----------------------------: | ------------------ | --------------------- |
| Leader lidera    |                  Corr lead-lag | positiva / nula    | seguir / ajustar      |
| Top corr aporta  |           diferencia vs random | positiva / nula    | mantener / cambiar    |
| Market cap sirve | hit rate vs leader estadístico | mejor / peor       | mantener / reemplazar |
| Edge factors     |                señales por día | razonable / escaso | ajustar               |
| Drift posterior  |                 forward return | positivo / nulo    | pasar a backtest      |

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys
import polars as pl
import pyarrow
import gc
from itertools import combinations
sys.path.append(str(Path("..")))
from src.ingest.open_parquet import abrir_parquet_data_pandas 

In [2]:
df_candidate_pairs = pd.read_csv('..\data\candidate_pairs.csv') 

<>:1: SyntaxWarning: invalid escape sequence '\d'
<>:1: SyntaxWarning: invalid escape sequence '\d'
C:\Users\avazq\AppData\Local\Temp\ipykernel_15336\3771880710.py:1: SyntaxWarning: invalid escape sequence '\d'
  df_candidate_pairs = pd.read_csv('..\data\candidate_pairs.csv')


In [3]:
df_candidate_pairs

,stock_a,stock_b,sector
0,CCEP,KDP,BEVERAGES
1,CCEP,PEP,BEVERAGES
2,KDP,PEP,BEVERAGES
3,AMGN,GILD,"BIOLOGICAL PRODUCTS, (NO DISGNOSTIC SUBSTANCES)"
4,CHTR,CMCSA,CABLE & OTHER PAY TELEVISION SERVICES
...,...,...,...
185,SHOP,TEAM,SERVICES-PREPACKAGED SOFTWARE
186,SHOP,TTWO,SERVICES-PREPACKAGED SOFTWARE
187,SNPS,TEAM,SERVICES-PREPACKAGED SOFTWARE
188,SNPS,TTWO,SERVICES-PREPACKAGED SOFTWARE


## Research Decision: Correlation Definition

For pair ranking we define:

- Observation frequency: 1-minute
- Variable: log returns
- Correlation: Pearson correlation
- Window: 60 trading days (23,460 observations)

Lead-lag relationships will be evaluated separately in Hypothesis 1 following Lo & MacKinlay (1990).

The correct workflow would be:

 1. Read each cleared stock by symbol

 2. For each symbol: 
     - calculate log_return_1m = log(close_t / close_t-1)

 3. For each pair in candidate_pairs:
     - merge stock_a and stock_b by date

4. Calculate the 60-day rolling correlation
     - window = 60 × 391 = 23,460 observations

5. For each day:
     - obtain the most recent available rolling correlation

6. Rank all pairs by correlation

7. Select the Top 50

In [4]:
DATA_DIR = Path("../data/rth_1min_by_symbol")
OUT_DIR = Path("../data/rth_1min_log_returns_by_symbol")
OUT_DIR.mkdir(exist_ok=True)

def add_log_returns(symbol_file):
    df = pd.read_parquet(symbol_file)
    df = df.sort_values("date").copy()
    df["log_return_1m"] = np.log(df["close"] / df["close"].shift(1))
    return df

In [ ]:
# save parquets with log returns
for file in DATA_DIR.glob("*_rth_1min.parquet"):
    symbol = file.name.replace("_rth_1min.parquet", "")

    df_ret = add_log_returns(file)

    df_ret.to_parquet(OUT_DIR / f"{symbol}_rth_1min_returns.parquet",index=False)

In [5]:
sample = pd.read_parquet(OUT_DIR / "NVDA_rth_1min_returns.parquet")

sample[["date", "symbol", "close", "log_return_1m"]].head()

,date,symbol,close,log_return_1m
0,2021-06-01 09:30:00,NVDA,16.251499,NaN
1,2021-06-01 09:31:00,NVDA,16.256500,0.000308
2,2021-06-01 09:32:00,NVDA,16.221300,-0.002168
3,2021-06-01 09:33:00,NVDA,16.180799,-0.002500
4,2021-06-01 09:34:00,NVDA,16.153601,-0.001682


Pair Selection Methodology

For each trading day t:

1. Use the previous 60 trading days of 1-minute log returns.
2. Compute Pearson correlation for every candidate pair.
3. Rank all candidate pairs by correlation.
4. Select the Top 50 pairs.
5. Use this Top 50 universe for signal generation during day t.

In [ ]:
RETURNS_DIR = Path("../data/rth_1min_log_returns_by_symbol")
WINDOW_DAYS = 60
BARS_PER_DAY = 391
WINDOW_SIZE = WINDOW_DAYS * BARS_PER_DAY


def load_pair_returns(stock_a, stock_b, returns_dir=RETURNS_DIR):
    df_a = pd.read_parquet(returns_dir / f"{stock_a}_rth_1min_returns.parquet")
    df_b = pd.read_parquet(returns_dir / f"{stock_b}_rth_1min_returns.parquet")

    df_pair = (
        df_a[["date", "log_return_1m"]]
        .rename(columns={"log_return_1m": f"ret_{stock_a}"})
        .merge(
            df_b[["date", "log_return_1m"]].rename(columns={"log_return_1m": f"ret_{stock_b}"}),
            on="date",
            how="inner"
        )
        .sort_values("date")
        .dropna()
        .reset_index(drop=True)
    )

    df_pair["trading_day"] = df_pair["date"].dt.date

    return df_pair

def compute_daily_rolling_corr_for_pair(stock_a, stock_b):
    df_pair = load_pair_returns(stock_a, stock_b)

    corr_col = "rolling_corr_60d"

    df_pair[corr_col] = (
        df_pair[f"ret_{stock_a}"]
        .rolling(WINDOW_SIZE)
        .corr(df_pair[f"ret_{stock_b}"])
    )

    daily_corr = (
        df_pair
        .groupby("trading_day", as_index=False)
        .tail(1)
        [["trading_day", corr_col]]
        .copy()
    )

    daily_corr["stock_a"] = stock_a
    daily_corr["stock_b"] = stock_b

    return daily_corr[["trading_day", "stock_a", "stock_b", corr_col]]


test_corr = compute_daily_rolling_corr_for_pair("NVDA", "AMD")

test_corr.head(70)

,trading_day,stock_a,stock_b,rolling_corr_60d
389,2021-06-01,NVDA,AMD,NaN
780,2021-06-02,NVDA,AMD,NaN
1171,2021-06-03,NVDA,AMD,NaN
1562,2021-06-04,NVDA,AMD,NaN
1953,2021-06-07,NVDA,AMD,NaN
...,...,...,...,...
25804,2021-09-01,NVDA,AMD,0.539634
26195,2021-09-02,NVDA,AMD,0.538799
26586,2021-09-03,NVDA,AMD,0.537889
26977,2021-09-07,NVDA,AMD,0.537677


In [10]:
test_corr["rolling_corr_60d"].isna().sum(), len(test_corr)

(np.int64(60), 1176)

## run for 160 candidate pears and build daily_pair_correlation

In [11]:
def compute_daily_rolling_corr_for_all_pairs(candidate_pairs):
    results = []

    for i, row in candidate_pairs.iterrows():
        stock_a = row["stock_a"]
        stock_b = row["stock_b"]

        print(f"{i+1}/{len(candidate_pairs)} - {stock_a}-{stock_b}")

        pair_corr = compute_daily_rolling_corr_for_pair(stock_a, stock_b)
        pair_corr["sector"] = row["sector"]

        results.append(pair_corr)

    return pd.concat(results, ignore_index=True)

In [12]:
daily_pair_correlations = compute_daily_rolling_corr_for_all_pairs(df_candidate_pairs)

1/190 - CCEP-KDP
2/190 - CCEP-PEP
3/190 - KDP-PEP
4/190 - AMGN-GILD
5/190 - CHTR-CMCSA
6/190 - FTNT-PANW
7/190 - STX-WDC
8/190 - EXC-XEL
9/190 - PCAR-TSLA
10/190 - ALNY-INSM
11/190 - ALNY-REGN
12/190 - ALNY-VRTX
13/190 - INSM-REGN
14/190 - INSM-VRTX
15/190 - REGN-VRTX
16/190 - AMZN-PDD
17/190 - COST-WMT
18/190 - ADI-AMAT
19/190 - ADI-AMD
20/190 - ADI-ASML
21/190 - ADI-AVGO
22/190 - ADI-INTC
23/190 - ADI-MCHP
24/190 - ADI-MPWR
25/190 - ADI-MRVL
26/190 - ADI-MU
27/190 - ADI-NVDA
28/190 - ADI-NXPI
29/190 - ADI-TXN
30/190 - AMAT-AMD
31/190 - AMAT-ASML
32/190 - AMAT-AVGO
33/190 - AMAT-INTC
34/190 - AMAT-MCHP
35/190 - AMAT-MPWR
36/190 - AMAT-MRVL
37/190 - AMAT-MU
38/190 - AMAT-NVDA
39/190 - AMAT-NXPI
40/190 - AMAT-TXN
41/190 - AMD-ASML
42/190 - AMD-AVGO
43/190 - AMD-INTC
44/190 - AMD-MCHP
45/190 - AMD-MPWR
46/190 - AMD-MRVL
47/190 - AMD-MU
48/190 - AMD-NVDA
49/190 - AMD-NXPI
50/190 - AMD-TXN
51/190 - ASML-AVGO
52/190 - ASML-INTC
53/190 - ASML-MCHP
54/190 - ASML-MPWR
55/190 - ASML-MRVL
56/190

In [20]:
daily_pair_correlations.head(62)

,trading_day,stock_a,stock_b,rolling_corr_60d,sector
0,2021-12-31,CCEP,KDP,NaN,BEVERAGES
1,2022-01-03,CCEP,KDP,NaN,BEVERAGES
2,2022-01-04,CCEP,KDP,NaN,BEVERAGES
3,2022-01-05,CCEP,KDP,NaN,BEVERAGES
4,2022-01-06,CCEP,KDP,NaN,BEVERAGES
...,...,...,...,...,...
57,2022-03-24,CCEP,KDP,NaN,BEVERAGES
58,2022-03-25,CCEP,KDP,NaN,BEVERAGES
59,2022-03-28,CCEP,KDP,NaN,BEVERAGES
60,2022-03-29,CCEP,KDP,0.408533,BEVERAGES


In [14]:
daily_pair_correlations.shape

(169641, 5)

In [15]:
daily_pair_correlations["rolling_corr_60d"].isna().sum()

np.int64(10210)

In [18]:
daily_pair_correlations.to_parquet("../data/daily_pair_correlations_60d.parquet",index=False)

## Hypothesis 1 — Does the Leader Actually Lead?

### Objective

The strategy assumes that, within each selected pair, the larger market-cap stock acts as the **leader** and the smaller market-cap stock acts as the **follower**.

This hypothesis tests whether leader returns contain information about future follower returns.

### Hypothesis

For each Top-50 pair selected by rolling 60-day correlation:

$ r_{leader,t} \rightarrow r_{follower,t+k} $

where \(k\) represents future lags such as 1, 5, 15 and 30 minutes.

### Test

For each pair and trading day:

1. Define leader as the stock with the larger market capitalization.
2. Define follower as the other stock.
3. Compute lagged correlations between:
   - leader return at time $ t $
   - follower return at time $ t+k $

### Decision Rule

Evidence supports the hypothesis if:

- lead-lag correlations are positive on average, and
- they are stronger than the reverse direction.

In [21]:
RETURNS_DIR = Path("../data/rth_1min_log_returns_by_symbol")
UNIVERSE_PATH = Path("../data/universe_metadata.csv")
CORR_PATH = Path("../data/daily_pair_correlations_60d.parquet")

In [ ]:
# First we built the daily Top 50
daily_pair_correlations = pd.read_parquet(CORR_PATH)

daily_top50_pairs = (daily_pair_correlations.dropna(subset=["rolling_corr_60d"]).sort_values(["trading_day", "rolling_corr_60d"], ascending=[True, False])
    .groupby("trading_day", as_index=False)
    .head(50)
    .reset_index(drop=True)
)

daily_top50_pairs.head()
daily_top50_pairs.to_csv("../data/daily_top50_pairs.csv", index=False)

,trading_day,stock_a,stock_b,rolling_corr_60d,sector
0,2021-08-25,GOOG,GOOGL,0.714106,"SERVICES-COMPUTER PROGRAMMING, DATA PROCESSING..."
1,2021-08-25,ADI,TXN,0.677198,SEMICONDUCTORS & RELATED DEVICES
2,2021-08-25,MCHP,NXPI,0.675279,SEMICONDUCTORS & RELATED DEVICES
3,2021-08-25,ADI,MCHP,0.656578,SEMICONDUCTORS & RELATED DEVICES
4,2021-08-25,MCHP,TXN,0.652763,SEMICONDUCTORS & RELATED DEVICES


In [ ]:
# we added leader/follower 
universe_df = pd.read_csv(UNIVERSE_PATH)

market_cap_map = universe_df.set_index("symbol")["market_cap"].to_dict()

def assign_leader_follower(row):
    a = row["stock_a"]
    b = row["stock_b"]

    if market_cap_map[a] >= market_cap_map[b]:
        return pd.Series({
            "leader": a,
            "follower": b,
            "leader_market_cap": market_cap_map[a],
            "follower_market_cap": market_cap_map[b]
        })
    else:
        return pd.Series({
            "leader": b,
            "follower": a,
            "leader_market_cap": market_cap_map[b],
            "follower_market_cap": market_cap_map[a]
        })

leader_follower_cols = daily_top50_pairs.apply(assign_leader_follower, axis=1)

daily_top50_pairs = pd.concat(
    [daily_top50_pairs, leader_follower_cols],
    axis=1
)

daily_top50_pairs.head()

,trading_day,stock_a,stock_b,rolling_corr_60d,sector,leader,follower,leader_market_cap,follower_market_cap
0,2021-08-25,GOOG,GOOGL,0.714106,"SERVICES-COMPUTER PROGRAMMING, DATA PROCESSING...",GOOG,GOOGL,4.022414e+12,4.018794e+12
1,2021-08-25,ADI,TXN,0.677198,SEMICONDUCTORS & RELATED DEVICES,TXN,ADI,2.021884e+11,1.565761e+11
2,2021-08-25,MCHP,NXPI,0.675279,SEMICONDUCTORS & RELATED DEVICES,NXPI,MCHP,5.709487e+10,4.227916e+10
3,2021-08-25,ADI,MCHP,0.656578,SEMICONDUCTORS & RELATED DEVICES,ADI,MCHP,1.565761e+11,4.227916e+10
4,2021-08-25,MCHP,TXN,0.652763,SEMICONDUCTORS & RELATED DEVICES,TXN,MCHP,2.021884e+11,4.227916e+10


In [24]:
# function to measure leader/ follower in a pair 
def compute_pair_lead_lag(leader, follower, lags=(1, 5, 15, 30)):
    df_leader = pd.read_parquet(RETURNS_DIR / f"{leader}_rth_1min_returns.parquet")
    df_follower = pd.read_parquet( RETURNS_DIR / f"{follower}_rth_1min_returns.parquet")

    df_pair = (df_leader[["date", "log_return_1m"]].rename(columns={"log_return_1m": "leader_ret"}).merge(df_follower[["date", "log_return_1m"]]
            .rename(columns={"log_return_1m": "follower_ret"}),on="date",how="inner").dropna().sort_values("date").reset_index(drop=True))

    results = {}

    for lag in lags:
        results[f"leader_to_follower_lag_{lag}m"] = (df_pair["leader_ret"].corr(df_pair["follower_ret"].shift(-lag)))

        results[f"follower_to_leader_lag_{lag}m"] = (df_pair["follower_ret"].corr(df_pair["leader_ret"].shift(-lag)))

    return results

In [25]:
unique_top_pairs = (daily_top50_pairs[["leader", "follower"]].drop_duplicates().reset_index(drop=True))

lead_lag_results = []

for i, row in unique_top_pairs.iterrows():
    leader = row["leader"]
    follower = row["follower"]

    print(f"{i+1}/{len(unique_top_pairs)} - {leader}->{follower}")

    metrics = compute_pair_lead_lag(leader, follower)

    lead_lag_results.append({"leader": leader,"follower": follower,**metrics})

lead_lag_df = pd.DataFrame(lead_lag_results)

lead_lag_df.head()

1/118 - GOOG->GOOGL
2/118 - TXN->ADI
3/118 - NXPI->MCHP
4/118 - ADI->MCHP
5/118 - TXN->MCHP
6/118 - XEL->EXC
7/118 - TXN->NXPI
8/118 - ADI->NXPI
9/118 - AMAT->ADI
10/118 - MU->AMAT
11/118 - AMAT->NXPI
12/118 - AMAT->MCHP
13/118 - AMAT->TXN
14/118 - AVGO->TXN
15/118 - AVGO->ADI
16/118 - MSFT->ADBE
17/118 - ADBE->CDNS
18/118 - NVDA->AMD
19/118 - ADI->MRVL
20/118 - SNPS->CDNS
21/118 - MRVL->MCHP
22/118 - AMAT->MRVL
23/118 - TXN->MRVL
24/118 - AVGO->MCHP
25/118 - MU->ADI
26/118 - AMGN->GILD
27/118 - MRVL->NXPI
28/118 - AVGO->AMAT
29/118 - MU->NXPI
30/118 - AVGO->NXPI
31/118 - CDNS->ADSK
32/118 - MU->MCHP
33/118 - MU->TXN
34/118 - ASML->AMAT
35/118 - MSFT->CDNS
36/118 - AVGO->MRVL
37/118 - ASML->NXPI
38/118 - ADBE->SNPS
39/118 - ADBE->ADSK
40/118 - MU->MRVL
41/118 - ASML->MCHP
42/118 - INTU->ADBE
43/118 - INTU->CDNS
44/118 - ASML->ADI
45/118 - MSFT->INTU
46/118 - AVGO->MU
47/118 - MSFT->ADSK
48/118 - INTC->TXN
49/118 - AVGO->ASML
50/118 - SNPS->ADSK
51/118 - NVDA->MRVL
52/118 - AMD->MRVL
53

,leader,follower,leader_to_follower_lag_1m,follower_to_leader_lag_1m,leader_to_follower_lag_5m,follower_to_leader_lag_5m,leader_to_follower_lag_15m,follower_to_leader_lag_15m,leader_to_follower_lag_30m,follower_to_leader_lag_30m
0,GOOG,GOOGL,-0.002154,0.002210,-0.001580,-0.002835,0.004973,0.002925,-0.000945,-0.001186
1,TXN,ADI,0.018909,0.006584,-0.001070,-0.005206,0.003961,0.003309,0.000441,-0.000253
2,NXPI,MCHP,0.026492,0.065470,-0.002995,0.001926,0.006047,0.005297,0.003722,-0.001147
3,ADI,MCHP,0.007694,0.023130,-0.003904,0.000891,0.008018,0.003521,0.002485,-0.000271
4,TXN,MCHP,0.012853,0.019628,-0.005619,0.001461,0.005759,0.001548,0.001218,-0.001207


In [26]:
lead_lag_df.describe()

,leader_to_follower_lag_1m,follower_to_leader_lag_1m,leader_to_follower_lag_5m,follower_to_leader_lag_5m,leader_to_follower_lag_15m,follower_to_leader_lag_15m,leader_to_follower_lag_30m,follower_to_leader_lag_30m
count,118.000000,118.000000,118.000000,118.000000,118.000000,118.000000,118.000000,118.000000
mean,0.013757,0.011282,-0.002295,-0.000671,0.003171,0.002743,-0.000047,-0.000056
std,0.014552,0.016358,0.003764,0.002535,0.002532,0.002396,0.002188,0.001857
min,-0.009427,-0.008809,-0.009524,-0.006223,-0.003917,-0.002325,-0.005615,-0.005032
25%,0.003111,-0.001059,-0.004859,-0.002238,0.001643,0.001052,-0.001230,-0.001142
50%,0.009431,0.004788,-0.002598,-0.000428,0.003019,0.002665,0.000097,0.000075
75%,0.024002,0.019490,0.000039,0.001193,0.004752,0.004596,0.001383,0.001286
max,0.058169,0.065470,0.008571,0.005650,0.009813,0.008546,0.004808,0.004162


In [27]:
lead_lag_df[
    [
        "leader_to_follower_lag_1m",
        "follower_to_leader_lag_1m"
    ]
].mean()

leader_to_follower_lag_1m    0.013757
follower_to_leader_lag_1m    0.011282
dtype: float64

In [28]:
(
    lead_lag_df["leader_to_follower_lag_1m"]
    >
    lead_lag_df["follower_to_leader_lag_1m"]
).mean()

np.float64(0.5677966101694916)

In [ ]:
lead_lag_df.to_parquet("../data/hypothesis_1_lead_lag_results.parquet", index=False)

### Preliminary Result

The market-cap rule shows weak evidence of directional leadership.

Results:

- Average leader→follower lag-1 correlation: 0.0138
- Average follower→leader lag-1 correlation: 0.0113
- Leader outperformed follower in 56.8% of pairs

Interpretation:

The evidence is consistent with a mild lead-lag effect in the expected direction, although the magnitude appears weak. Market capitalization may capture part of the leadership structure, but likely does not fully explain information transmission across pairs.

## Hypothesis 2 — Does Rolling Correlation Add Value?

### Objective

The strategy ranks candidate pairs using a 60-day rolling correlation and trades only the Top 50 pairs.

This hypothesis evaluates whether the rolling-correlation ranking identifies pairs with stronger lead-lag relationships than randomly selected same-sector pairs.

### Null Hypothesis

Top correlation pairs do not exhibit stronger lead-lag effects than random same-sector pairs.

### Alternative Hypothesis

Top correlation pairs exhibit stronger lead-lag effects than random same-sector pairs.

### Evaluation Metric

Lag-1 lead-lag correlation:

$ corr(
r_leader(t),
r_follower(t+1)
) $

### Decision Rule

If Top50 pairs show higher average lead-lag correlation than random same-sector pairs, the rolling-correlation filter adds value.

In [ ]:
# Random benchmark:
# take randomly 50 pairs
# mesure lead-lag average 
#  repeat 1,000 times 
# If the Top50 falls within the 80th, 90th, and 95th percentiles of the random distribution, then rolling correlation does contribute.

observed_top50_mean = (lead_lag_df["leader_to_follower_lag_1m"].mean())

# When the leader moves in one minute, the follower tends to move slightly in the same direction one minute later.

np.float64(0.013757028668868656)

In [ ]:
n_top_pairs = len(lead_lag_df)
n_top_pairs

# what does it mean ? 118 that appeared at least once in a daily Top 50 list
# candidate_pairs = 190, But the Top 50 ranking changed day by day.
# 118 pairs were considered "interesting" at some point by the strategy.


118

In [ ]:
# QUESTION :  is 0.013757 special? or Does any random selection of pairs produce something similar?

# First we need to calculate the same lead-lag for all 190 candidate pairs
candidate_pairs_with_roles = df_candidate_pairs.copy()
roles = candidate_pairs_with_roles.apply(assign_leader_follower,axis=1)
candidate_pairs_with_roles = pd.concat([candidate_pairs_with_roles, roles],axis=1)
candidate_pairs_with_roles.head()
candidate_pairs_with_roles.shape

(190, 7)

In [35]:
# Now we're going to build the base dataset for the benchmark
candidate_leadlag_results = []

for i, row in candidate_pairs_with_roles.iterrows():

    leader = row["leader"]
    follower = row["follower"]

    print(f"{i+1}/{len(candidate_pairs_with_roles)} " f"{leader}->{follower}")

    metrics = compute_pair_lead_lag(leader,follower)

    candidate_leadlag_results.append({"leader": leader,"follower": follower,"sector": row["sector"],**metrics})

candidate_leadlag_df = pd.DataFrame(candidate_leadlag_results)
candidate_leadlag_df.shape

1/190 CCEP->KDP
2/190 PEP->CCEP
3/190 PEP->KDP
4/190 AMGN->GILD
5/190 CMCSA->CHTR
6/190 PANW->FTNT
7/190 WDC->STX
8/190 XEL->EXC
9/190 TSLA->PCAR
10/190 ALNY->INSM
11/190 REGN->ALNY
12/190 VRTX->ALNY
13/190 REGN->INSM
14/190 VRTX->INSM
15/190 VRTX->REGN
16/190 AMZN->PDD
17/190 WMT->COST
18/190 AMAT->ADI
19/190 AMD->ADI
20/190 ASML->ADI
21/190 AVGO->ADI
22/190 INTC->ADI
23/190 ADI->MCHP
24/190 ADI->MPWR
25/190 ADI->MRVL
26/190 MU->ADI
27/190 NVDA->ADI
28/190 ADI->NXPI
29/190 TXN->ADI
30/190 AMD->AMAT
31/190 ASML->AMAT
32/190 AVGO->AMAT
33/190 INTC->AMAT
34/190 AMAT->MCHP
35/190 AMAT->MPWR
36/190 AMAT->MRVL
37/190 MU->AMAT
38/190 NVDA->AMAT
39/190 AMAT->NXPI
40/190 AMAT->TXN
41/190 ASML->AMD
42/190 AVGO->AMD
43/190 AMD->INTC
44/190 AMD->MCHP
45/190 AMD->MPWR
46/190 AMD->MRVL
47/190 MU->AMD
48/190 NVDA->AMD
49/190 AMD->NXPI
50/190 AMD->TXN
51/190 AVGO->ASML
52/190 ASML->INTC
53/190 ASML->MCHP
54/190 ASML->MPWR
55/190 ASML->MRVL
56/190 ASML->MU
57/190 NVDA->ASML
58/190 ASML->NXPI
59/190 AS

(190, 11)

In [37]:
# we already have the correct dataset to perform the benchmark.
candidate_leadlag_df.head(2)

,leader,follower,sector,leader_to_follower_lag_1m,follower_to_leader_lag_1m,leader_to_follower_lag_5m,follower_to_leader_lag_5m,leader_to_follower_lag_15m,follower_to_leader_lag_15m,leader_to_follower_lag_30m,follower_to_leader_lag_30m
0,CCEP,KDP,BEVERAGES,0.004110,0.026390,-0.002551,0.003729,0.003299,0.002008,0.001963,0.002590
1,PEP,CCEP,BEVERAGES,0.030483,0.010354,0.004506,-0.002036,-0.000826,0.005631,0.001255,0.001996


In [39]:
# Bootstrap
bootstrap_means = []

for _ in range(10000):

    sample = candidate_leadlag_df.sample(n=n_top_pairs,replace=False)

    bootstrap_means.append(sample["leader_to_follower_lag_1m"].mean())

bootstrap_means = np.array(bootstrap_means)
bootstrap_means.mean()

np.float64(0.01432898852757192)

In [40]:
bootstrap_means.std()

np.float64(0.0016228793520580618)

In [ ]:
np.percentile(bootstrap_means,[5, 25, 50, 75, 95])


array([0.01168811, 0.01318236, 0.01430491, 0.01547147, 0.01706619])

In [42]:
(bootstrap_means < observed_top50_mean).mean()

np.float64(0.3654)

 Do pairs selected using 60-day rolling correlation have a greater lead-lag effect than randomly chosen pairs? 
- We found no evidence that it did.

Do Top50 pairs selected by rolling 60-day correlation exhibit stronger lead-lag effects than randomly selected same-sector pairs?

<b> Result: </b> No meaningful improvement was observed.

The average leader-to-follower lag correlation of the Top50 universe (0.0138) fell near the center of the bootstrap distribution generated from random candidate pairs.

Conclusion

Rolling correlation appears useful for identifying co-moving assets, but does not appear to identify stronger lead-lag relationships by itself.